Tek tek deneme yapmak yerine hatanın nereden geldiğini ölçmeyi ve hangi yolların açık hangilerinin kapalı olduğunu anlamaya çalışıyoruz.

Özet:

1. Hatanın %70'i cold-start segmentinden geliyor: (test satırlarının %22'si).
   Profilli taraf ~0.66 RMSLE, cold taraf ~1.87.

2. Cold hatasının %61'i, cold satırların sadece %3.8'inden geliyor: tüketimi
   tamamen sıfır olan trafolar. Bir sıfır satırı, normal bir satırın ~95 katı
   kare hata üretiyor.

3. Bu sıfır trafolar tahmin edilemiyor. Sızıntısız kurulan sınıflandırıcı
   AUC 0.53 veriyor (yazı-tura). Güç, lokasyon, giriş tarihi, gözlem süresi;
   hiçbiri ayırt etmiyor.

4. Cold-start'a yapılan dört ayrı müdahale de Kaggleda olumsuz üçü de lokal validationda iyileşme gösterdiği halde.

5. Sebebi: lokal validation'ın cold segmenti testinkini temsil
   etmiyor. Test cold trafolarının %68'i 2026-05-11'de toplu devreye giren,
   güç medyanı 630 kVA'lık trafolar. Lokal cold trafolar ise güç medyanı
   400 kVA olan, ortalama 20 gün gözlenmiş dağınık trafolar. Trainde bu toplu
   giriş olayının benzeri yok, dolayısıyla simüle edilemiyor.

Sonuç: cold-start bu veriyle çözülebilir görünmüyor; kalan hata tahmin
edilemeyen sıfır trafolardan oluşan bir taban. İyileştirme bütçesi profilli
tarafa (hatanın %30'u) ayrılmalı.

In [1]:
import pandas as pd
import numpy as np

RAW = '../data/raw/'
train = pd.read_csv(RAW + 'train.csv', parse_dates=['tarih'])
test = pd.read_csv(RAW + 'test.csv', parse_dates=['tarih'])

for kolon, sira in [('il', 0), ('bolge', 1), ('ilce', 2)]:
    train[kolon] = train['lokasyon'].str.split('>').str[sira]
    test[kolon] = test['lokasyon'].str.split('>').str[sira]
train['ilce'] = train['ilce'].fillna('YOK')
test['ilce'] = test['ilce'].fillna('YOK')

cold_trafolar = set(test['tanim']) - set(train['tanim'])
test['cold'] = test['tanim'].isin(cold_trafolar)

print('train', train.shape, '| test', test.shape)
print('train tarih:', train.tarih.min().date(), '->', train.tarih.max().date())
print('test  tarih:', test.tarih.min().date(), '->', test.tarih.max().date())
print()
print(f'train trafo {train.tanim.nunique()} | test trafo {test.tanim.nunique()}')
print(f'cold-start trafo: {len(cold_trafolar)} '
      f'-> test satırlarının %{test.cold.mean()*100:.2f} si')

train (1226237, 8) | test (714688, 9)
train tarih: 2025-01-01 -> 2026-03-31
test  tarih: 2026-04-01 -> 2026-07-31

train trafo 5344 | test trafo 7036
cold-start trafo: 2024 -> test satırlarının %22.16 si


Hata bütçesi: RMSLE'yi iki segmente ayırmak

RMSLE karesi satır bazında ortalanabildiği için toplam skoru segmentlere
ayırabiliriz:

Burada cold satır payı (0.2216), profilli segmentin RMSLE'si, cold
segmentinki. profillileri lokal validation'dan biliyoruz
(~0.66), dolayısıyla Kaggle skorundan coldsuzları geri hesaplayabiliyoruz.

Bunu doğrulayan bir kanıt da var: `sub_10` ve `sub_11` profilli satırlarda
birebir aynı çıktı, sadece cold satırlarda farklı. Yani aradaki skor farkı
tamamen cold segmentinden geliyor.

In [ ]:
W = test['cold'].mean() # cold satır payı
KAGGLE_EN_IYI = 1.05487 # sub_10
P_TAHMIN = 0.66 # profilli RMSLE, lokal validation ölçümü


def cold_geri_hesap(kaggle_skoru, p=P_TAHMIN, w=W):
    return np.sqrt((kaggle_skoru ** 2 - (1 - w) * p ** 2) / w) #toplam skor - cold segment RMSLE


def kaggle_tahmin(cold_rmsle, p=P_TAHMIN, w=W):
    return np.sqrt((1 - w) * p ** 2 + w * cold_rmsle ** 2) #Segment RMSLE'lerinden toplam skoru tahmin eder.


C_TAHMIN = cold_geri_hesap(KAGGLE_EN_IYI)
prof_pay = (1 - W) * P_TAHMIN ** 2 / KAGGLE_EN_IYI ** 2

print(f'en iyi skor {KAGGLE_EN_IYI} icin:')
print(f'  profilli: satır payı %{(1-W)*100:.1f}  RMSLE {P_TAHMIN:.2f}  '
      f'hata payı %{prof_pay*100:.1f}')
print(f'  cold    : satır payı %{W*100:.1f}  RMSLE {C_TAHMIN:.2f}  '
      f'hata payı %{(1-prof_pay)*100:.1f}')
print()
print('hangi iyileşme ne kadar skor getirir:')
for ad, c, p in [('cold 1.87 -> 1.60', 1.60, P_TAHMIN),
                 ('cold 1.87 -> 1.40', 1.40, P_TAHMIN),
                 ('profilli 0.66 -> 0.63', C_TAHMIN, 0.63),
                 ('profilli 0.66 -> 0.60', C_TAHMIN, 0.60)]:
    print(f'  {ad:24s} -> {kaggle_tahmin(c, p):.4f}')

en iyi skor 1.05487 icin:
  profilli: satır payı %77.8  RMSLE 0.66  hata payı %30.5
  cold    : satır payı %22.2  RMSLE 1.87  hata payı %69.5

hangi iyileşme ne kadar skor getirir:
  cold 1.87 -> 1.60        -> 0.9520
  cold 1.87 -> 1.40        -> 0.8794
  profilli 0.66 -> 0.63    -> 1.0405
  profilli 0.66 -> 0.60    -> 1.0266


Sıfır tüketim: hatanın asıl kaynağı

RMSLE logaritmik olduğu için sıfıra yakın değerler orantısız ağırlık taşır.
Gerçek tüketim 0 iken 1000 kWh tahmin edersek hata log(1001) = 6.9, karesi
47 olur. Normal bir satırda kare hata ~0.5'tir.

Aşağıdaki dekompozisyon, lokal simülasyonda cold segmentinin hatasını gerçek
tüketim seviyesine göre parçalıyor.

In [ ]:
ORIGIN = pd.Timestamp('2025-12-31')
gecmis = train[train.tarih <= ORIGIN]
gelecek = train[train.tarih > ORIGIN]
sim_cold = sorted(set(gelecek.tanim) - set(gecmis.tanim))
sim = gelecek[gelecek.tanim.isin(sim_cold)].copy()

# basit referans tahmin: guc x geçmişin ortalama (tuketim/guc) oranı
g = gecmis[gecmis.guc > 0]
oran = np.expm1(np.log1p(g.tuketim / g.guc).mean())
sim['tahmin'] = sim['guc'] * oran
sim['kare_hata'] = (np.log1p(sim.tahmin) - np.log1p(sim.tuketim)) ** 2
toplam = sim.kare_hata.sum()

sim['seviye'] = pd.cut(sim.tuketim, [-0.1, 0.5, 100, 1000, 10000, np.inf],
                       labels=['tam sıfır', '0-100', '100-1k', '1k-10k', '>10k'])
d = sim.groupby('seviye', observed=True).agg(satir=('kare_hata', 'size'),
                                             hata_payi=('kare_hata', lambda x: x.sum() / toplam))
d['satir_payi'] = d.satir / len(sim)

print(f'simüle cold segment RMSLE: {np.sqrt(sim.kare_hata.mean()):.4f}')
print(f'{len(sim_cold)} cold trafo, {len(sim):,} satır\n')
print(d[['satir', 'satir_payi', 'hata_payi']].round(4).to_string())

sifir_trafo = sim.groupby('tanim')['tuketim'].max().pipe(lambda s: set(s[s == 0].index))
temiz = sim[~sim.tanim.isin(sifir_trafo)]
print(f'\ntamamen sıfır trafo: {len(sifir_trafo)} / {sim.tanim.nunique()}')
print(f'bunlar cikarilirsa cold RMSLE: {np.sqrt(sim.kare_hata.mean()):.4f} '
      f'-> {np.sqrt(temiz.kare_hata.mean()):.4f}')

simüle cold segment RMSLE: 1.6555
873 cold trafo, 26,707 satır

           satir  satir_payi  hata_payi
seviye                                 
tam sıfır   1098      0.0411     0.6839
0-100       1581      0.0592     0.0839
100-1k      9417      0.3526     0.0862
1k-10k     14435      0.5405     0.1381
>10k         176      0.0066     0.0080

tamamen sıfır trafo: 34 / 873
bunlar cikarilirsa cold RMSLE: 1.6555 -> 1.0467


## 3. Sıfır trafolar tahmin edilebilir mi? Hayır.

Yukarıdaki tablo net bir hedef gösteriyor: sıfır trafoları bilebilseydik cold
RMSLE 1.66'dan 1.03'e inerdi. Bunun için bir sınıflandırıcı denedik.

İlk denemede AUC 0.977 çıktı ancak kurulumda sızıntı vardı: eğitim etiketleri hedef dönemi de
kapsayan bir pencereden hesaplanıyordu ve gun (gözlem süresi) özelliği
eğitimde uzun, hedefte kısa pencereden geliyordu. Pencereleri eşitleyip hedef
dönemi dışarıda bırakınca AUC 0.53'e düştü, yani sinyal yok.

Bu, çalışma boyunca tekrar eden bir ders: bir sonuç fazla iyi görünüyorsa önce
sızıntı aranmalı.

In [4]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

PENCERE = pd.DateOffset(months=3)   # egitim ve hedef icin ayni gozlem penceresi


def cold_trafo_tablosu(origin):
    """Origin'de ilk kez gorulen trafolar, sabit pencereyle ozetlenir."""
    o = pd.Timestamp(origin)
    d = train[(train.tarih >= o) & (train.tarih < o + PENCERE)
              & (~train.tanim.isin(set(train.loc[train.tarih < o, 'tanim'])))]
    tf = d.groupby('tanim').agg(guc=('guc', 'first'), il=('il', 'first'),
                                bolge=('bolge', 'first'), ilce=('ilce', 'first'),
                                gun=('tarih', 'size'), ilk=('tarih', 'min'),
                                sifir=('tuketim', lambda x: (x == 0).mean()))
    tf['y'] = (tf.sifir >= 0.99).astype(int)
    tf['ilk_ay'] = tf.ilk.dt.month
    return tf


egitim_tf = pd.concat([cold_trafo_tablosu(o) for o in
                       ['2025-04-01', '2025-06-01', '2025-08-01', '2025-10-01']])
hedef_tf = cold_trafo_tablosu('2026-01-01')

for k in ['il', 'bolge', 'ilce']:
    kats = pd.Categorical(pd.concat([egitim_tf[k], hedef_tf[k]])).categories
    egitim_tf[k + '_k'] = pd.Categorical(egitim_tf[k], categories=kats).codes
    hedef_tf[k + '_k'] = pd.Categorical(hedef_tf[k], categories=kats).codes
OZ = ['guc', 'gun', 'ilk_ay', 'il_k', 'bolge_k', 'ilce_k']

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, verbose=-1)
clf.fit(egitim_tf[OZ], egitim_tf['y'])
hedef_tf['p'] = clf.predict_proba(hedef_tf[OZ])[:, 1]

print(f'eğitim {len(egitim_tf)} trafo (%{egitim_tf.y.mean()*100:.1f} sıfır) | '
      f'hedef {len(hedef_tf)} trafo (%{hedef_tf.y.mean()*100:.1f} sıfır)')
print(f'AUC = {roc_auc_score(hedef_tf.y, hedef_tf.p):.4f}   '
      f'(0.5 = tamamen rastgele)')
print(f'en riskli 20 trafonun {hedef_tf.nlargest(20, "p").y.sum()}/20 si gerçekten sıfır')

eğitim 2738 trafo (%7.5 sıfır) | hedef 873 trafo (%3.9 sıfır)
AUC = 0.5303   (0.5 = tamamen rastgele)
en riskli 20 trafonun 1/20 si gerçekten sıfır


## 4. Denenen cold-start çözümleri ve Kaggle sonuçları

Dört yaklaşım denendi. Üçü lokal validation'da iyileşme gösterdi, hiçbiri
Kaggle'a taşınmadı.

| Deneme | Yöntem | Lokal | Kaggle |
|---|---|---|---|
| `sub_06` | Profil dropout %80 | iyileşme | 1.10514 |
| `sub_11` | Cold satırlara grup + güç oranı istatistiği | cold 1.89 → 1.64 | 1.05725 |
| `sub_14` | Komşu profili ödünç verme + 3 seed | iki dönemde de iyileşme | 1.06631 |
| — | Cold için ayrı uzman model | kötüleşti | denenmedi |
| **`sub_10`** | **Müdahale yok (referans)** | — | **1.05487** |

Ayrıca test edilip elenen fikirler:

- **Tahminleri küçültmek (shrinkage).** Sıfır riskine karşı sigorta olarak
  log-tahmini $k$ ile çarpmak. $k=0.98$ marjinal, $k=0.95$ zaten kötüleştiriyor.
  Mevcut tahmin ölçeği optimal noktada.
- **Cold satırları eğitimde ağırlıklandırmak.** Ağırlık arttıkça cold da
  profilli de bozuluyor.
- **`tanim` kolonundan bilgi çıkarmak.** Numerik ID, isim bilgisi yok. ID öneki
  sıfır oranıyla zayıf ilişkili (%3.3–%7.3 arası), kullanılabilir sinyal değil.
- **Yeni trafo rampası.** Yeni giren trafolar 1. günden itibaren normal
  seviyede çalışıyor (referansın %100'ü), rampa yok. Yani "yeni trafo az
  tüketir" varsayımı geçersiz.

## 5. Neden hiçbiri tutmadı: lokal cold ≠ test cold

Lokal validation toplam skoru çok iyi tahmin ediyor (1.0672 tahmin, 1.0549
gerçek). Cold segmentinin *seviyesini* de doğru buluyor (~1.89 lokal, ~1.87
geri hesap). Ama cold segmentindeki *iyileştirmeler* transfer olmuyor.

Sebep, iki cold popülasyonunun farklı olması. Test'in cold trafolarının %68'i
tek bir günde, 2026-05-11'de toplu devreye giriyor; bunlar güç medyanı 630 kVA
olan ve 122 günün 82'sinde görülen büyük trafolar. Lokal validation'ın cold
trafoları ise güç medyanı 400 kVA olan, ortalama 20 gün gözlenmiş dağınık
girişler.

Train içinde bu ölçekte bir toplu giriş olayı yok (en yakını 2026-03-26'da 329
trafo, ki o da train'in son 6 gününe denk geliyor). Dolayısıyla test'in cold
segmentini simüle edecek bir validation kurmak bu veriyle mümkün değil.

In [5]:
te_cold = test[test.cold]
ilk_gorulme = te_cold.groupby('tanim')['tarih'].min()
print('test cold trafolarının devreye giriş tarihleri (ilk 5):')
print(ilk_gorulme.value_counts().head(5).to_string())

toplu = set(ilk_gorulme[ilk_gorulme == pd.Timestamp('2026-05-11')].index)
tt = te_cold.groupby('tanim').agg(guc=('guc', 'first'), gun=('id', 'size'))
tt['toplu_giris'] = tt.index.isin(toplu)

lokal = train[train.tanim.isin(sim_cold)].groupby('tanim').agg(
    guc=('guc', 'first'), gun=('tarih', 'size'))

print()
print(f'{"popülasyon":34s} {"trafo":>6s} {"güç medyan":>11s} {"gün medyan":>11s}')
print(f'{"TEST cold - 2026-05-11 toplu":34s} {tt.toplu_giris.sum():6d} '
      f'{tt[tt.toplu_giris].guc.median():11.0f} {tt[tt.toplu_giris].gun.median():11.0f}')
print(f'{"TEST cold - diğerleri":34s} {(~tt.toplu_giris).sum():6d} '
      f'{tt[~tt.toplu_giris].guc.median():11.0f} {tt[~tt.toplu_giris].gun.median():11.0f}')
print(f'{"LOKAL validation cold":34s} {len(lokal):6d} '
      f'{lokal.guc.median():11.0f} {lokal.gun.median():11.0f}')
print()
print(f'toplu giriş grubunun cold satırlarındaki payı: '
      f'%{tt[tt.toplu_giris].gun.sum()/tt.gun.sum()*100:.1f}')

test cold trafolarının devreye giriş tarihleri (ilk 5):
tarih
2026-05-11    1326
2026-04-30     114
2026-05-03     105
2026-05-07      88
2026-05-13      36

popülasyon                          trafo  güç medyan  gün medyan
TEST cold - 2026-05-11 toplu         1326         630          82
TEST cold - diğerleri                 698         400          86
LOKAL validation cold                 873         400          20

toplu giriş grubunun cold satırlarındaki payı: %68.4


## 6. Açık kalan yol: profilli taraf

Cold segmenti kapalıyken hatanın %30'unu oluşturan profilli taraf hâlâ açık.
Orada ölçülmüş ve seed'ler arasında tutarlı bir kazanç var.

Kritik gözlem: model kapasitesi arttıkça profilli taraf iyileşiyor, cold
taraf çöküyor. Tek model kullandığımız için bugüne kadar cold'u korumak adına
kapasiteyi 87 ağaçta tutmuşuz. İki tahmin yolunu ayırırsak bu kısıt kalkıyor.

| ağaç / yaprak | profilli RMSLE | cold RMSLE |
|---|---|---|
| 87 / 31 | 0.6592 ± 0.004 | 1.904 ± 0.008 |
| 200 / 31 | 0.6497 ± 0.001 | 2.161 ± 0.034 |
| 300 / 63 | 0.6334 ± 0.010 | 2.530 ± 0.028 |
| 500 / 63 | 0.6964 ± 0.055 | 2.806 ± 0.104 |

sub_13_ikili_kapasite.csv bu fikri uyguluyor: cold satırlar 87 ağaçlı
modelden (yani sub_10 ile birebir aynı), profilli satırlar 300/63'ten
geliyor. Cold tarafı sabit tuttuğu için temiz bir A/B testi. Beklenen ~1.042.





